# 01 — Crater Floor Acoustic Processing

This notebook is the cleaned version of the final Crater Floor batch-processing workflow.

It starts from the retained Crater Floor WAV files and reproduces the processing used to create
`LIBS_acoustic_frequency_results.csv`.

The original Crater Floor FITS → WAV conversion code was not retained, so that earlier conversion step
is outside the scope of this notebook.

Expected full-dataset output: 209 rows × 19 columns.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import find_peaks, windows
from scipy.stats import linregress

## Paths

Put the original Crater Floor WAV files in:

`data/raw/crater_floor_wav/`

The processed CSV will be written to:

`data/intermediate/LIBS_acoustic_frequency_results.csv`

In [ ]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAV_DIR = PROJECT_ROOT / "data" / "raw" / "crater_floor_wav"
OUTPUT_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "LIBS_acoustic_frequency_results.csv"

print("WAV directory:", WAV_DIR.resolve())
print("Output:", OUTPUT_PATH.resolve())

## Processing settings

These values are retained from the final Crater Floor processing block in the original `FFT.ipynb`.

In [ ]:
# Expected timing of the first LIBS shot
EXPECTED_SHOT_TIME = 0.073
NEXT_SHOT_START = 0.133

# Analysis window
RESPONSE_WINDOW = 0.010

# Energy Decay Curve fit
FIT_DB_TOP = -7
FIT_DB_BOTTOM = -15

# C2 cutoff
C2_CUTOFF = 0.002

# Onset / peak detection
SEARCH_HALF_WIDTH = 0.005
NOISE_WINDOW = 0.003
SMOOTH_WINDOW_S = 0.00005
THRESHOLD_SIGMA = 4

MIN_PEAK_DISTANCE_S = 0.010
PROMINENCE_SIGMA = 3
HEIGHT_SIGMA = 4

BACKTRACK_WINDOW_S = 0.003
BACKTRACK_NOISE_S = 0.005
BACKTRACK_SIGMA = 4
MIN_RUN_S = 0.00005

# Initial Crater Floor flag threshold
FLAG_LAG_MS = 0.4

# FFT frequency bands
USABLE_BAND = (1000, 50000)
LOW_BAND = (1000, 10000)
HIGH_BAND = (10000, 30000)

## Frequency-domain metrics

In [ ]:
FFT_COLUMNS = [
    "Spectral Centroid (Hz)",
    "Spectral Bandwidth (Hz)",
    "Peak Frequency (Hz)",
    "Rolloff 85% (Hz)",
    "Low Power 1-10 kHz",
    "High Power 10-30 kHz",
    "High/Low Ratio",
    "High Frequency Fraction",
    "Total FFT Power",
]


def empty_fft_metrics():
    return {column: np.nan for column in FFT_COLUMNS}


def compute_fft_metrics(segment, fs):
    # Remove DC component and apply the same Hann window used in the original code.
    segment = segment - np.mean(segment)

    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(len(seg_w), d=1 / fs)
    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (
        (freqs >= USABLE_BAND[0])
        & (freqs <= USABLE_BAND[1])
    )
    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) == 0:
        return empty_fft_metrics()

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum
    bandwidth = np.sqrt(
        np.sum(((f - centroid) ** 2) * p) / p_sum
    )
    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)
    rolloff_85 = f[
        np.where(cumulative >= 0.85 * p_sum)[0][0]
    ]

    low_mask = (
        (freqs >= LOW_BAND[0])
        & (freqs < LOW_BAND[1])
    )
    high_mask = (
        (freqs >= HIGH_BAND[0])
        & (freqs < HIGH_BAND[1])
    )

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    high_low_ratio = (
        high_power / low_power
        if low_power > 0
        else np.nan
    )
    high_freq_fraction = (
        high_power / total_power
        if total_power > 0
        else np.nan
    )

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_low_ratio,
        "High Frequency Fraction": high_freq_fraction,
        "Total FFT Power": total_power,
    }

## Process one WAV

The signal is searched around the expected first-shot time. The detected peak is backtracked to the first
sustained envelope crossing, which is used as the acoustic onset. A 10 ms segment beginning at the onset is
then used for the EDC, C2, and FFT calculations.

In [ ]:
def process_wav(wav_path):
    fs, x = wavfile.read(wav_path)
    x = x.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    abs_x = np.abs(x)

    # Smoothed absolute-amplitude envelope
    smooth_n = max(1, int(round(SMOOTH_WINDOW_S * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")

    # Search around the expected first LIBS shot
    search_start = EXPECTED_SHOT_TIME - SEARCH_HALF_WIDTH
    search_end = EXPECTED_SHOT_TIME + SEARCH_HALF_WIDTH

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    # Noise window immediately before the search region
    inoise1 = isearch0
    inoise0 = max(
        0,
        int(round((search_start - NOISE_WINDOW) * fs)),
    )

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    threshold = noise_mean + THRESHOLD_SIGMA * noise_std
    search_env = env[isearch0:isearch1]

    min_peak_distance = int(round(MIN_PEAK_DISTANCE_S * fs))
    min_prominence = PROMINENCE_SIGMA * noise_std
    min_height = noise_mean + HEIGHT_SIGMA * noise_std

    peaks, _ = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance,
    )

    found_onset = len(peaks) > 0
    flagged = False
    lag_ms = np.nan

    if found_onset:
        # Original batch code used the first qualifying peak.
        peak_index = isearch0 + peaks[0]
        peak_time = peak_index / fs

        # Backtrack from peak to identify onset
        backtrack_samples = int(
            round(BACKTRACK_WINDOW_S * fs)
        )
        back_start = max(
            isearch0,
            peak_index - backtrack_samples,
        )

        noise_back_start = max(
            0,
            back_start
            - int(round(BACKTRACK_NOISE_S * fs)),
        )
        local_noise = env[noise_back_start:back_start]

        if len(local_noise) > 0:
            onset_threshold = (
                np.mean(local_noise)
                + BACKTRACK_SIGMA * np.std(local_noise)
            )
        else:
            onset_threshold = threshold

        search_back_env = env[back_start:peak_index]
        above = search_back_env > onset_threshold

        min_run_samples = max(
            1,
            int(round(MIN_RUN_S * fs)),
        )

        onset_index = None
        for i in range(
            len(above) - min_run_samples + 1
        ):
            if np.all(
                above[i:i + min_run_samples]
            ):
                onset_index = back_start + i
                break

        if onset_index is None:
            onset_index = peak_index
            flagged = True

        response_start = onset_index / fs
        response_stop = (
            response_start + RESPONSE_WINDOW
        )

        lag_ms = (
            (peak_index - onset_index)
            / fs
            * 1000
        )

        if lag_ms > FLAG_LAG_MS:
            flagged = True

        # Prevent overlap with the next LIBS shot
        if response_stop > NEXT_SHOT_START:
            response_stop = NEXT_SHOT_START

        i0 = onset_index
        i1 = int(round(response_stop * fs))

        segment = x[i0:i1]
        t = np.arange(len(segment)) / fs

        # Schroeder Energy Decay Curve
        energy = segment ** 2
        edc = np.cumsum(energy[::-1])[::-1]
        edc_norm = edc / np.max(edc)
        edc_db = 10 * np.log10(
            edc_norm + 1e-20
        )

        fit_mask = (
            (edc_db <= FIT_DB_TOP)
            & (edc_db >= FIT_DB_BOTTOM)
        )

        if np.sum(fit_mask) < 2:
            slope = np.nan
            r2 = np.nan
            drop_time = np.nan
        else:
            (
                slope,
                _,
                r_value,
                _,
                _,
            ) = linregress(
                t[fit_mask],
                edc_db[fit_mask],
            )

            r2 = r_value ** 2
            db_drop = abs(
                FIT_DB_BOTTOM - FIT_DB_TOP
            )
            drop_time = (
                db_drop / abs(slope)
                if slope != 0
                else np.nan
            )

        # C2
        i_c = int(round(C2_CUTOFF * fs))
        early_energy = np.sum(energy[:i_c])
        late_energy = np.sum(energy[i_c:])

        C2 = (
            10
            * np.log10(
                early_energy / late_energy
            )
            if late_energy > 0
            else np.nan
        )

        fft_metrics = compute_fft_metrics(
            segment,
            fs,
        )

    else:
        response_start = np.nan
        peak_time = np.nan
        slope = np.nan
        r2 = np.nan
        drop_time = np.nan
        C2 = np.nan
        fft_metrics = empty_fft_metrics()

    return {
        "File Name": wav_path.name,
        "Flagged": flagged,
        "Sample Rate (Hz)": fs,
        "Onset (s)": response_start,
        "Peak Time (s)": peak_time,
        "Peak-Onset Lag (ms)": lag_ms,
        "Slope (dB/s)": slope,
        "R^2": r2,
        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,
        **fft_metrics,
    }

## Batch processing

In [ ]:
wav_files = sorted(WAV_DIR.glob("*.wav"))

if not wav_files:
    raise FileNotFoundError(
        f"No WAV files found in {WAV_DIR.resolve()}"
    )

results = []

for i, wav_path in enumerate(
    wav_files,
    start=1,
):
    row = process_wav(wav_path)
    results.append(row)

    print(
        f"[{i:03d}/{len(wav_files):03d}] "
        f"{wav_path.name} | "
        f"flagged={row['Flagged']}"
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("\nSaved:", OUTPUT_PATH)
print("Shape:", results_df.shape)
print("\nFlag counts:")
print(
    results_df["Flagged"]
    .value_counts(dropna=False)
)

## Expected result

With the complete original Crater Floor WAV collection, the historical output was:

- 209 observations
- 19 columns
- 161 initially unflagged
- 48 flagged by the batch-processing flag

Later manual review/QC was broader than this single automatic flag and is handled separately in
`02_crater_floor_qc_recovery.ipynb`.